# 📊 Notebook 4: Evaluation — CRF vs mBERT

**Side-by-side comparison of Traditional vs Transformer NER on Hindi**

This notebook:
1. Loads saved results from CRF and mBERT
2. Builds the main comparison table
3. Plots entity-level F1 comparison
4. Analyzes performance by entity length
5. Computes entity confusion matrices
6. Generates all figures for the report

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation import (
    build_comparison_table,
    plot_f1_comparison,
    plot_confusion_matrix,
    compute_entity_confusion_matrix,
    performance_by_entity_length,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Saved Results

In [ ]:
# Load metrics
with open('../results/crf_test_metrics.json', 'r') as f:
    crf_metrics = json.load(f)
    
with open('../results/mbert_test_metrics.json', 'r') as f:
    mbert_metrics = json.load(f)

# Load predictions
with open('../results/predictions/crf_predictions.json', 'r', encoding='utf-8') as f:
    crf_predictions = json.load(f)

with open('../results/predictions/mbert_predictions.json', 'r', encoding='utf-8') as f:
    mbert_predictions = json.load(f)

print(f"CRF  — Overall F1: {crf_metrics['overall_f1']:.4f}")
print(f"mBERT — Overall F1: {mbert_metrics['overall_f1']:.4f}")

## 2. Main Comparison Table

**This table goes directly into your report and presentation.**

In [ ]:
comparison_df = build_comparison_table(crf_metrics, mbert_metrics)

print("\n" + "=" * 80)
print("CRF vs mBERT — ENTITY-LEVEL COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Save for report
comparison_df.to_csv('../results/tables/crf_vs_mbert_comparison.csv', index=False)
print("\n✓ Saved to results/tables/crf_vs_mbert_comparison.csv")

## 3. F1 Comparison Bar Chart

In [ ]:
plot_f1_comparison(
    crf_metrics, 
    mbert_metrics,
    save_path='../results/figures/f1_comparison.png'
)

## 4. Entity Confusion Matrices

In [ ]:
# CRF confusion matrix
crf_confusion = compute_entity_confusion_matrix(crf_predictions)
print("CRF Entity Confusion Matrix:")
print(crf_confusion)
plot_confusion_matrix(crf_confusion, 'CRF', 
                      save_path='../results/figures/crf_confusion.png')

# mBERT confusion matrix
mbert_confusion = compute_entity_confusion_matrix(mbert_predictions)
print("\nmBERT Entity Confusion Matrix:")
print(mbert_confusion)
plot_confusion_matrix(mbert_confusion, 'mBERT',
                      save_path='../results/figures/mbert_confusion.png')

## 5. Performance by Entity Length

In [ ]:
# CRF performance by entity length
crf_length = performance_by_entity_length(crf_predictions)
crf_length.columns = [c if c not in ['Accuracy'] else 'CRF Accuracy' for c in crf_length.columns]

# mBERT performance by entity length
mbert_length = performance_by_entity_length(mbert_predictions)
mbert_length.columns = [c if c not in ['Accuracy'] else 'mBERT Accuracy' for c in mbert_length.columns]

print("\nCRF — Performance by Entity Length:")
print(crf_length.to_string(index=False))

print("\nmBERT — Performance by Entity Length:")
print(mbert_length.to_string(index=False))

In [ ]:
# Plot entity length comparison
fig, ax = plt.subplots(figsize=(8, 5))

# Merge the two DataFrames on entity length
crf_acc = performance_by_entity_length(crf_predictions)
mbert_acc = performance_by_entity_length(mbert_predictions)

lengths = sorted(set(crf_acc['Entity Length (tokens)'].tolist() + 
                     mbert_acc['Entity Length (tokens)'].tolist()))

crf_vals = []
mbert_vals = []
for l in lengths:
    crf_row = crf_acc[crf_acc['Entity Length (tokens)'] == l]
    mbert_row = mbert_acc[mbert_acc['Entity Length (tokens)'] == l]
    crf_vals.append(crf_row['Accuracy'].values[0] if len(crf_row) > 0 else 0)
    mbert_vals.append(mbert_row['Accuracy'].values[0] if len(mbert_row) > 0 else 0)

import numpy as np
x = np.arange(len(lengths))
width = 0.35

ax.bar(x - width/2, crf_vals, width, label='CRF', color='#4C72B0', alpha=0.85)
ax.bar(x + width/2, mbert_vals, width, label='mBERT', color='#DD8452', alpha=0.85)

ax.set_xlabel('Entity Length (tokens)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Performance by Entity Length', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(lengths)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/performance_by_length.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Where Models Disagree

Show sentences where CRF and mBERT give different predictions.

In [ ]:
# Find disagreements
disagreements = []
for crf_pred, mbert_pred in zip(crf_predictions, mbert_predictions):
    if crf_pred['pred_labels'] != mbert_pred['pred_labels']:
        # Check if at least one has an entity prediction
        has_entity = (any(l != 'O' for l in crf_pred['pred_labels']) or
                      any(l != 'O' for l in mbert_pred['pred_labels']))
        if has_entity:
            disagreements.append({
                'tokens': crf_pred['tokens'],
                'true': crf_pred['true_labels'],
                'crf': crf_pred['pred_labels'],
                'mbert': mbert_pred['pred_labels'],
            })

print(f"Total disagreements (with entities): {len(disagreements)}")
print("\nInteresting disagreements:")
print("=" * 70)

for d in disagreements[:5]:
    print(f"\nSentence: {' '.join(d['tokens'])}")
    print(f"{'Token':20s} {'True':10s} {'CRF':10s} {'mBERT':10s}")
    print("-" * 55)
    for t, true_l, crf_l, mbert_l in zip(d['tokens'], d['true'], d['crf'], d['mbert']):
        marker = '→ ' if crf_l != mbert_l else '  '
        print(f"{marker}{t:20s} {true_l:10s} {crf_l:10s} {mbert_l:10s}")